In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
import joblib

feature_extractor = Model(
    inputs=(base:=ResNet50(weights='imagenet', include_top=False,
                           input_shape=(224, 224, 3))).input,
    outputs=GlobalAveragePooling2D()(base.output)
)

model_rf = joblib.load('/content/drive/MyDrive/Colab Notebooks/random_forest_model.joblib')

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import preprocess_input

weight = 0

def predict_melanoma(image_path):
    global weight
    if not os.path.exists(image_path): 
        print(f"Không tìm thấy ảnh: {image_path}")
        return None
    img = load_img(image_path, target_size=(224, 224))
    plt.imshow(img); plt.axis('off'); plt.show()
    x = preprocess_input(np.expand_dims(img_to_array(img), 0))
    features = feature_extractor.predict(x, verbose=0).reshape(1, -1)
    pred = model_rf.predict(features)[0]
    prob = np.max(model_rf.predict_proba(features)) * 100
    print(f"Dự đoán: {pred} (Xác suất: {prob:.2f}%)")
    if pred == 1: weight += 2
    return weight

In [ ]:
image_path = '/content/drive/MyDrive/Colab Notebooks/Dataset_Melanoma/img/ISIC_0024329.jpg'
predict_melanoma(image_path)
print("Weight:", weight)

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import torch, cv2, numpy as np
from torchvision import transforms
import segmentation_models_pytorch as smp

model = smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1)
model.load_state_dict(torch.load('/content/drive/MyDrive/Colab Notebooks/unet_melanoma_epoch.pth', map_location='cpu'))
model.eval()

def predict_and_show(image_path):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (256, 256))
    x = transforms.ToTensor()(img_resized).unsqueeze(0)
    with torch.no_grad():
        mask = (torch.sigmoid(model(x)).squeeze().cpu().numpy() > 0.5).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    outlined = img_resized.copy()
    cv2.drawContours(outlined, contours, -1, (255, 0, 0), 2)
    return outlined 

In [ ]:
def extract_asymmetry(mask, threshold=0.2):
    ys, xs = np.where(mask > 0)
    if not len(xs) or not len(ys): return 0
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    tumor = mask[y0:y1+1, x0:x1+1]
    h, w = tumor.shape
    s = max(h, w)
    pad = np.zeros((s, s), dtype=np.uint8)
    pad[(s-h)//2:(s-h)//2 + h, (s-w)//2:(s-w)//2 + w] = tumor
    c = s // 2
    l, r = pad[:, :c], pad[:, c:]
    m = min(l.shape[1], r.shape[1])
    diff = abs((l[:, :m] > 0).sum() - (r[:, :m] > 0).sum())
    total = max((l[:, :m] > 0).sum(), (r[:, :m] > 0).sum())
    return int(total > 0 and diff / total > threshold)

In [ ]:
def get_pred_mask(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    x = transforms.ToTensor()(cv2.resize(img, (256, 256))).unsqueeze(0)
    with torch.no_grad():
        mask = (torch.sigmoid(model(x)).squeeze().cpu().numpy() > 0.5).astype(np.uint8)
    return mask

if extract_asymmetry(get_pred_mask(test_image_path), threshold=0.2):
    weight += 1

print("Weight =", weight)

In [ ]:
def extract_border_irregularity_area_based(mask, threshold_ratio=0.02, visualize=False):
    c,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not c: return 0
    cnt = max(c, key=cv2.contourArea)
    a_cnt, a_hull = cv2.contourArea(cnt), cv2.contourArea(cv2.convexHull(cnt))
    ratio = (a_hull - a_cnt) / a_hull if a_hull else 0
    irr = int(ratio > threshold_ratio)
    if visualize:
        print(f"Area Contour={a_cnt}, Hull={a_hull}, Defect={a_hull - a_cnt}, Ratio={ratio:.4f} → Irregular={irr}")
    return irr

In [ ]:
# Tính border irregularity
border_irregularity = extract_border_irregularity_area_based(pred_mask, threshold_ratio=0.03, visualize=True)
print("Border Irregularity =", border_irregularity)

if border_irregularity == 1:
    weight += 1
    print(weight)

In [ ]:
from sklearn.cluster import KMeans
import numpy as np, matplotlib.pyplot as plt
from webcolors import rgb_to_hex

def extract_color_variation(image, n_colors=3, visualize=False):
    kmeans = KMeans(n_clusters=n_colors, random_state=42, n_init=10).fit(image.reshape(-1, 3))
    colors = kmeans.cluster_centers_.astype(int)
    uniq = np.unique(kmeans.labels_)
    val = int(len(uniq) >= 2)

    if visualize:
        plt.subplot(121); plt.imshow(image); plt.axis('off'); plt.title("Original")
        color_img = np.zeros((100, 100*n_colors, 3), dtype=np.uint8)
        for i,c in enumerate(colors): color_img[:, i*100:(i+1)*100] = c
        plt.subplot(122); plt.imshow(color_img); plt.axis('off'); plt.title("Dominant Colors")
        plt.show()
        for i,c in enumerate(colors):
            print(f"Color {i+1}: RGB={c}, Hex={rgb_to_hex(tuple(c))}")
        print(f"Unique colors: {len(uniq)}  →  Variation={val}")

    return val

In [ ]:
pred_mask, image = get_pred_mask(test_image_path)
color_var = extract_color_variation(image, visualize=True)

if color_var == 1:
    weight += 1

print("Weight:", weight)

In [ ]:
if weight >= 3:
    final_prediction = 1  # Melanoma
else:
    final_prediction = 0  # Không phải melanoma

print("Dự đoán khối u là melanoma?" , "Có" if final_prediction == 1 else "Không")


In [ ]:
import os

def process_folder(folder):
    for f in [x for x in os.listdir(folder) if x.lower().endswith(('.jpg','.jpeg','.png'))]:
        path = os.path.join(folder, f)
        print(f"\n=== Xử lý ảnh: {f} ===")
        w = 0
        predict_melanoma(path)  # hàm này đã tự tăng weight toàn cục nếu cần
        mask, img = get_pred_mask(path)
        if extract_asymmetry(mask): w += 1; print("→ Asymmetry: YES")
        if extract_border_irregularity_area_based(mask, 0.03): w += 1; print("→ Border Irregularity: YES")
        if extract_color_variation(img): w += 1; print("→ Color Variation: YES")
        print(f"✅ Final prediction for {f}: {'MELANOMA' if w >= 3 else 'Không phải melanoma'}")
        print(f"→ Tổng điểm weight = {w}")

In [ ]:
def process_folder_silent(folder_path, output_csv='results.csv'):
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    results = []

    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        weight = 0

        predict_melanoma(image_path)

        pred_mask, image = get_pred_mask(image_path)

        if extract_asymmetry(pred_mask) == 1:
            weight += 1

        if extract_border_irregularity_area_based(pred_mask, threshold_ratio=0.03) == 1:
            weight += 1

        if extract_color_variation(image) == 1:
            weight += 1

        final_prediction = 1 if weight >= 3 else 0
        results.append([image_file, final_prediction])

    with open(output_csv, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['image_name', 'final_prediction'])
        writer.writerows(results)

    print(f"Đã quét và xử lý {len(image_files)} ảnh.")

In [ ]:
process_folder_silent('/content/drive/MyDrive/Colab Notebooks/Dataset_Melanoma/img', '/content/drive/MyDrive/Colab Notebooks/Dataset_Melanoma/test-results.csv')

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

m = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Dataset_Melanoma/metamelanoma.csv')
r = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Dataset_Melanoma/test-results.csv')

m['image'] = m['image'].str.strip().str.lower()
r['image'] = r['image_name'].str.replace('.jpg','',regex=False).str.strip().str.lower()

m = m.drop_duplicates('image').sort_values('image')
r = r.drop_duplicates('image').sort_values('image')

df = m.merge(r[['image','final_prediction']], on='image')
print(f"Số dòng metamelanoma: {len(m)}, test-results: {len(r)}, sau merge: {len(df)}")

acc = (df['melanoma']==df['final_prediction']).mean()
print(f"Accuracy: {acc*100:.2f}%")
print(classification_report(df['melanoma'], df['final_prediction'],
                            target_names=["Không melanoma","Melanoma"]))